# LFW 03. Compressed vectors and identity templates

## 예상 소요 시간 (LFW 13,195개·로컬 PostgreSQL 기준)

| 실행 모드 | 예상 시간 | 주요 작업 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run과 이전 phase 확인 준비 |
| `EXECUTE_STAGE=True` | 약 10~40분 | PCA/PQ 저장, test/calibration identity template 저장, index 확인 |

> 30초 heartbeat와 배치 진행률을 출력합니다.

목표: 이미지별 PCA/PQ 산출물뿐 아니라 실제 검색 대상인 identity template을 `template_embedding_512`와 `template_embedding_256`에 저장합니다. Test와 calibration scope는 서로 다른 `protocol_name`으로 분리합니다.

> **재시작 규칙**: Kernel Restart 후 Run All을 사용합니다. 같은 run/model의 동일 행은 건너뜁니다. protocol/config가 바뀌면 00부터 새 run을 시작하십시오.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('03 materialization/index', heartbeat_seconds=30)


## Plan

- 02의 고정 PCA/PQ artifact로 이미지별 압축 표현을 materialize합니다.
- 00 test protocol과 calibration split에서 identity template을 생성합니다.
- 원본 512D template과 PCA-256 retrieval template을 동일 scope로 DB에 저장합니다.
- PQ code는 pgvector 검색 대상에 포함하지 않습니다.

In [2]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest

def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for path in sorted(attempts.glob('A*/phase_manifest.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}.')
    return max(completed)

preflight = {'execute_stage': EXECUTE_STAGE, 'run_dir_resolved': str(RUN_DIR),
             'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file())}
preflight


{'execute_stage': True,
 'run_dir_resolved': 'D:\\ronbun\\runs\\lfw\\2026\\07\\14\\20260714-R004-16a5cbeb_thesis3_lfw_face_search_v1',
 'run_manifest_exists': True}

## Execute and record

`template_embedding_512`는 원본 ArcFace 평균 template, `template_embedding_256`은 PCA retrieval template입니다. reconstructed 512D는 certificate 계산에만 사용합니다.

In [3]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import pandas as pd
    from research.compression import PCACompressor, PQCompressor, PCA_256
    from research.database import create_database_engine, load_database_settings
    from research.experiments import (
        LFWTemplateScope,
        build_lfw_certification_inputs,
        materialize_compressed_embeddings,
        materialize_lfw_templates,
        protocol_frames as as_protocol_frames,
    )
    from research.protocols import build_calibration_protocol
    from research.runtime.hashing import sha256_file

    with PROGRESS.step('run/input 및 00·02 artifact 검증', expected='10초 미만'):
        run, run_manifest = attach_run(RUN_DIR)
        run.verify_inputs()
        run.verify_phase_artifacts('00_protocol_and_run_freeze')
        run.verify_phase_artifacts('02_compressor_fit')
        protocol_attempt = latest_completed_attempt(RUN_DIR, '00_protocol_and_run_freeze')
        protocol_suffix = f'A{protocol_attempt:03d}'
        protocol_dir = RUN_DIR / 'artifacts' / '00_protocol_and_run_freeze'
        test_protocol_frames = {
            role: pd.read_csv(protocol_dir / f'{role}_{protocol_suffix}.csv')
            for role in ('gallery', 'registered_probes', 'known_unknown_probes', 'unknown_unknown_probes')
        }
        compressor_attempt = latest_completed_attempt(RUN_DIR, '02_compressor_fit')
        compressor_suffix = f'A{compressor_attempt:03d}'
        artifact_dir = RUN_DIR / 'artifacts' / '02_compressor_fit'
        pca_path = artifact_dir / f'pca_256_{compressor_suffix}.joblib'
        pq_path = artifact_dir / f'pq_{compressor_suffix}.faiss'
        if not pca_path.is_file() or not pq_path.is_file():
            raise FileNotFoundError({'pca': str(pca_path), 'pq': str(pq_path)})
        pca_sha256 = sha256_file(pca_path)
        pca_model_uid = f'{pca_path.name}:{pca_sha256}'
        pca = PCACompressor.load(pca_path)
        pq = PQCompressor.load(pq_path)
        config = run_manifest['config']
        manifest = pd.read_csv(PROJECT_ROOT / config['dataset']['manifest_path'])
        development_paths = {
            str((PROJECT_ROOT / Path(str(path))).resolve())
            if not Path(str(path)).is_absolute()
            else str(Path(str(path)).resolve())
            for path in manifest.loc[manifest['split'].eq('development'), 'image_path']
        }
        if not development_paths:
            raise ValueError('development image paths are required for frozen normalization.')
        gallery_counts = test_protocol_frames['gallery'].groupby('identity_id').size()
        if gallery_counts.nunique() != 1:
            raise ValueError('LFW test gallery enrollment count must be uniform.')
        enrollment_target = int(gallery_counts.iloc[0])
        gallery_identity_count = int(test_protocol_frames['gallery']['identity_id'].nunique())
        calibration_protocol = build_calibration_protocol(
            manifest,
            split_name='calibration',
            gallery_identity_count=gallery_identity_count,
            enrollment_count=enrollment_target,
            seed=int(config['protocol']['split_seed']),
        )
        calibration_protocol_frames = as_protocol_frames(calibration_protocol)
        engine = create_database_engine(load_database_settings())

    with run.phase('03_compressed_materialization_and_index') as phase:
        suffix = f'A{phase.attempt:03d}'
        measurements_path = phase.attempt_dir / f'compression_measurements_{suffix}.csv'

        def report(message: str, details: dict[str, object]) -> None:
            PROGRESS.emit(message, **details)

        with PROGRESS.step('이미지별 PCA/PQ materialization', expected='10~30분'):
            summary = materialize_compressed_embeddings(
                engine,
                run_uid=run.run_id,
                pca=pca,
                pq=pq,
                pca_artifact_path=pca_path,
                pca_artifact_sha256=pca_sha256,
                pq_artifact_path=pq_path,
                pq_artifact_sha256=sha256_file(pq_path),
                development_image_paths=development_paths,
                measurements_path=measurements_path,
                batch_size=512,
                progress=report,
            )

        with PROGRESS.step('test/calibration template 생성 및 DB 저장', expected='1~10분'):
            test_bundle = build_lfw_certification_inputs(
                engine,
                run_uid=run.run_id,
                protocol_frames=test_protocol_frames,
                project_root=PROJECT_ROOT,
                compression_profile=PCA_256,
                pca=pca,
            )
            calibration_bundle = build_lfw_certification_inputs(
                engine,
                run_uid=run.run_id,
                protocol_frames=calibration_protocol_frames,
                project_root=PROJECT_ROOT,
                compression_profile=PCA_256,
                pca=pca,
                allow_empty_unknown_unknown=True,
            )
            test_scope = LFWTemplateScope(
                run_uid=run.run_id,
                protocol_name='lfw_test',
                model_uid=pca_model_uid,
                enrollment_target=enrollment_target,
            )
            calibration_scope = LFWTemplateScope(
                run_uid=run.run_id,
                protocol_name='lfw_calibration',
                model_uid=pca_model_uid,
                enrollment_target=enrollment_target,
            )
            template_summary = {
                'test': materialize_lfw_templates(
                    engine, bundle=test_bundle, scope=test_scope, progress=report
                ),
                'calibration': materialize_lfw_templates(
                    engine, bundle=calibration_bundle, scope=calibration_scope, progress=report
                ),
            }

        calibration_protocol_paths = []
        for role, frame in calibration_protocol_frames.items():
            output_path = phase.attempt_dir / f'calibration_{role}_{suffix}.csv'
            frame.to_csv(output_path, index=False, encoding='utf-8', lineterminator='\n')
            calibration_protocol_paths.append(output_path)
        summary['compressor_attempt'] = compressor_suffix
        summary['pca_model_uid'] = pca_model_uid
        summary['template_materialization'] = template_summary
        summary['test_protocol_coverage'] = test_bundle.coverage
        summary['calibration_protocol_coverage'] = calibration_bundle.coverage
        summary_path = phase.attempt_dir / f'materialization_summary_{suffix}.json'
        summary_path.write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        for artifact in [measurements_path, *calibration_protocol_paths, summary_path]:
            phase.publish_artifact(artifact)
        phase.record_counts(
            source_vectors=int(summary['counts']['source_vectors']),
            test_templates=int(template_summary['test']['template_count']),
            calibration_templates=int(template_summary['calibration']['template_count']),
        )
    PROGRESS.emit(
        '03 완료',
        source_vectors=summary['counts']['source_vectors'],
        test_templates=template_summary['test']['template_count'],
        calibration_templates=template_summary['calibration']['template_count'],
    )
    result = {'status': 'completed', 'run_id': run.run_id, **summary}
else:
    PROGRESS.emit('검토 모드 완료: DB 저장과 index 작업을 실행하지 않음', expected='1초 미만')
result

[23:28:09] 03 materialization/index | START run/input 및 00·02 artifact 검증 | elapsed=2s | expected=10초 미만
[23:28:12] 03 materialization/index | DONE run/input 및 00·02 artifact 검증 | elapsed=5s | step_elapsed=3s
[23:28:12] 03 materialization/index | START 이미지별 PCA/PQ materialization | elapsed=5s | expected=10~30분
[23:28:14] 03 materialization/index | development normalization scan | elapsed=7s | scanned=512 development_vectors=0
[23:28:14] 03 materialization/index | development normalization scan | elapsed=8s | scanned=1024 development_vectors=0
[23:28:15] 03 materialization/index | development normalization scan | elapsed=8s | scanned=1536 development_vectors=0
[23:28:15] 03 materialization/index | development normalization scan | elapsed=8s | scanned=2048 development_vectors=0
[23:28:15] 03 materialization/index | development normalization scan | elapsed=8s | scanned=2560 development_vectors=0
[23:28:15] 03 materialization/index | development normalization scan | elapsed=8s | scanned=30

{'status': 'completed',
 'run_id': '20260714-R004-16a5cbeb',
 'counts': {'source_vectors': 13195,
  'pca_inserted': 13195,
  'pca_skipped': 0,
  'pq_inserted': 13195,
  'pq_skipped': 0},
 'row_counts': {'embedding_512': 13195,
  'embedding_256': 13195,
  'embedding_pq': 13195},
 'error_normalization': {'pca_256': {'mean': 0.0002485459262970835,
   'std': 9.308511653216556e-05,
   'fit_count': 7742},
  'pq_auxiliary': {'mean': 0.0011110190534964204,
   'std': 0.0003286988940089941,
   'fit_count': 7742}},
 'measurements_path': 'D:\\ronbun\\runs\\lfw\\2026\\07\\14\\20260714-R004-16a5cbeb_thesis3_lfw_face_search_v1\\phases\\03_compressed_materialization_and_index\\attempts\\A001\\compression_measurements_A001.csv',
 'index_ensure_elapsed_seconds': 0.006178099996759556,
 'index_measurement_note': 'Time spent ensuring indexes exist; this is not a clean HNSW build-time measurement when indexes were created before row insertion.',
 'storage_bytes': {'embedding_512': 81780736,
  'embedding_256

## Next step

`template_embedding_512`와 `template_embedding_256`의 test/calibration scope별 행 수가 각각 gallery identity 수와 같아야 합니다. 04는 이 DB template만 검색합니다.